# Phase 0.5 — check the measuring instrument, not the router

**This notebook proves nothing about the router.** It answers one question: *if we run the
promised experiments, can the result ever be conclusive?*

The ship rule says: pass only if the router beats the baseline by more than **Δmin = 0.02**,
shown by a 95% confidence interval. A confidence interval has a width, and that width is a
property of the **data design** — how many queries, how many collections, how noisy each
collection's average is — not of the router. So we can compute it *today*, from scores
already on disk, by replaying the exact split-and-average procedure the plan precommitted.

Two numbers per design:

- **Resolution** — the CI half-width. If it is bigger than 0.02, every experiment ends
  "can't tell" *by construction*: a router truly 0.02 better, measured with a ±0.05
  instrument, produces intervals like [−0.03, +0.07] — straddling the bar every time.
- **Power** — if the router truly were 0.02 better, how often would this procedure say
  PASS? Power 0.02 = one time in fifty = a lottery ticket, not a measurement.

Cell map: (1) duplicate clusters — the correlation structure the intervals must respect;
(2) Design A = C1's grouped split, precommitted pooling; (3) the per-lane table that
*diagnoses* the width — most lanes have too few answerable test queries to deserve a full
vote; (4) the same deltas under three pooling rules, side by side — the decision cell;
(5) Design B = C3's hide-one-lane rotation. Machinery imported from
`scripts/feasibility_gate.py`; VERDICT.md cites these cells.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src" / "scripts"))

import numpy as np
import pandas as pd
from feasibility_gate import (
    CLUSTERS_PATH,
    DELTA_MIN,
    fit_and_deltas,
    grouped_split_random,
    pooled_bootstrap,
    power,
)

from hybrid_search_rrf_dataset.router import RouterExperiment

exp = RouterExperiment()
data = exp.load()
cluster_map = pd.read_parquet(CLUSTERS_PATH)  # built once by feasibility_gate.py
merged = data.merge(cluster_map, on=["dataset", "query_id"], how="left")
clusters = merged["cluster_id"]

sizes = clusters.value_counts()
dup_rows = int(sizes[sizes > 1].sum())
print(f"rows {len(merged):,} | clusters {clusters.nunique():,} | "
      f"rows in >1-member clusters {dup_rows:,} ({dup_rows / len(merged):.1%}) — d33 said ~5.5%")
print(f"largest cluster: {int(sizes.max())} queries")

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rows 46,142 | clusters 44,280 | rows in >1-member clusters 2,585 (5.6%) — d33 said ~5.5%
largest cluster: 189 queries


In [2]:
# Design A — C1's shape: grouped random split within lane (clusters never straddle),
# router vs per-lane train-selected best constant, answerable held-out rows.
train, test = grouped_split_random(merged, clusters)
deltas_a = fit_and_deltas(train, test, baseline="per_lane")
se_a = pooled_bootstrap(deltas_a, clusters)
print(f"lanes {deltas_a['dataset'].nunique()} | answerable test rows {len(deltas_a):,}")
print(f"precommitted lane-equal pooled: SE {se_a:.4f}, CI half-width ±{1.96 * se_a:.4f} "
      f"→ resolvable at {DELTA_MIN}: {'YES' if 1.96 * se_a < DELTA_MIN else 'NO'}")
print("power:", {d: round(power(se_a, d), 2) for d in (0.02, 0.03, 0.04, 0.05)})

lanes 41 | answerable test rows 7,597
precommitted lane-equal pooled: SE 0.0269, CI half-width ±0.0527 → resolvable at 0.02: NO
power: {0.02: 0.02, 0.03: 0.06, 0.04: 0.11, 0.05: 0.2}


In [3]:
# The diagnosis: per-lane deltas with their evidence. One-vote-per-lane lets 3-row lanes
# swing the pooled CI as hard as 1,500-row lanes.
lane = deltas_a.groupby("dataset")["delta"].agg(["mean", "count", "std"]).sort_values("mean")
lane.columns = ["lane_delta", "n_answerable", "row_std"]
print(f"between-lane std: {lane['lane_delta'].std():.3f}")
print(f"lanes with <100 answerable test rows: {(lane['n_answerable'] < 100).sum()} of {len(lane)}")
lane.round(3)

between-lane std: 0.153
lanes with <100 answerable test rows: 32 of 41


,lane_delta,n_answerable,row_std
dataset,,,
crumb-stack-exchange,-0.609,3,0.534
limit,-0.524,54,0.491
freshstack-godot,-0.249,11,0.373
bright-biology,-0.209,16,0.382
rarb-code,-0.205,36,0.399
freshstack-laravel,-0.197,29,0.331
crumb-clinical-trial,-0.194,4,0.328
bright-stackoverflow,-0.178,16,0.522
bright-psychology,-0.109,15,0.279


In [4]:
# Re-pooling variants of the SAME deltas — the decision Phase 1 needs before it can run.
# None of these is precommitted; choosing one is threshold-adjacent and gets a name on it.
big = lane[lane["n_answerable"] >= 100]
se_big = float(big["lane_delta"].std() / np.sqrt(len(big)))
se_row = float(deltas_a["delta"].std() / np.sqrt(len(deltas_a)))
variants = pd.DataFrame([
    {"pooling": "lane-equal, all lanes (precommitted)", "point": lane["lane_delta"].mean(),
     "SE": se_a, "half_width": 1.96 * se_a, "resolvable_at_0.02": 1.96 * se_a < DELTA_MIN},
    {"pooling": f"lane-equal, only >=100-row lanes ({len(big)})", "point": big["lane_delta"].mean(),
     "SE": se_big, "half_width": 1.96 * se_big, "resolvable_at_0.02": 1.96 * se_big < DELTA_MIN},
    {"pooling": "row-weighted, all rows", "point": deltas_a["delta"].mean(),
     "SE": se_row, "half_width": 1.96 * se_row, "resolvable_at_0.02": 1.96 * se_row < DELTA_MIN},
])
print("NOTE the sign of `point` in every powered variant: router − per-lane best constant.")
variants.round(4)

NOTE the sign of `point` in every powered variant: router − per-lane best constant.


,pooling,point,SE,half_width,resolvable_at_0.02
0,"lane-equal, all lanes (precommitted)",-0.0563,0.0269,0.0527,False
1,"lane-equal, only >=100-row lanes (9)",-0.0154,0.0060,0.0117,True
2,"row-weighted, all rows",-0.0196,0.0025,0.0048,True


In [5]:
# Design B — C3's shape: hide-one-lane rotation over every lane, router vs train-global
# constant. ~40 fits; takes a few minutes.
lane_deltas = []
for lane_name in sorted(merged["dataset"].unique()):
    d = fit_and_deltas(
        merged[merged["dataset"] != lane_name],
        merged[merged["dataset"] == lane_name],
        baseline="global",
    )
    if d.empty:
        print(f"{lane_name}: no answerable rows — excluded")
        continue
    lane_deltas.append(d)
deltas_b = pd.concat(lane_deltas)
se_b = pooled_bootstrap(deltas_b, clusters)
print(f"lanes {deltas_b['dataset'].nunique()} | answerable rows {len(deltas_b):,}")
print(f"precommitted lane-equal pooled: SE {se_b:.4f}, CI half-width ±{1.96 * se_b:.4f} "
      f"→ resolvable at {DELTA_MIN}: {'YES' if 1.96 * se_b < DELTA_MIN else 'NO'}")
print("power:", {d: round(power(se_b, d), 2) for d in (0.02, 0.03, 0.04, 0.05)})
rot = deltas_b.groupby("dataset")["delta"].agg(["mean", "count"]).sort_values("mean")
rot.round(3)

bright-aops: no answerable rows — excluded


lanes 41 | answerable rows 37,961
precommitted lane-equal pooled: SE 0.0323, CI half-width ±0.0634 → resolvable at 0.02: NO
power: {0.02: 0.02, 0.03: 0.05, 0.04: 0.09, 0.05: 0.15}


,mean,count
dataset,,
crumb-theorem-retrieval,-0.884,1
freshstack-godot,-0.174,62
bright-psychology,-0.121,63
bright-sustainable-living,-0.098,74
bright-biology,-0.094,79
bright-stackoverflow,-0.087,84
bright-robotics,-0.075,60
bright-earth-science,-0.066,98
freshstack-laravel,-0.063,150


## Reading

- **Precommitted estimator (lane-equal over all lanes): unresolvable in both designs** —
  half-widths ±0.05–0.06 against Δmin = 0.02, power 2%. Cause is in the per-lane table:
  most lanes carry <100 answerable held-out rows, and one vote per lane propagates their
  noise into the pooled CI.
- **Both powered re-poolings put the C1 point estimate below zero**: the router loses to
  the per-lane best constant on answerable rows. The historical +0.019 was against global
  const-dense; C1's bar (per-lane best constant) sits ~3.7 points higher on current data.
- Status: diagnostic (one grouped split, no Codex attack). The estimator amendment is a
  named decision recorded in VERDICT.md before Phase 1 formalizes any claim.